In [ ]:
import pandas as pd
import nltk
import kagglehub
import os

# 1. Tải các gói ngôn ngữ NLTK (Đã bổ sung punkt_tab để sửa lỗi LookupError)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# 2. Tải dataset từ Kaggle bằng kagglehub
print("Đang tải dữ liệu từ Kaggle...")
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

# 3. Tìm file CSV trong thư mục vừa tải về
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)

# 4. Đọc dữ liệu
df = pd.read_csv(full_path)
df_sample = df.head(1000).copy()

print(f"\n--- Hoàn tất! Đã nạp {len(df_sample)} dòng dữ liệu ---")
display(df_sample.head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Đang tải dữ liệu từ Kaggle...
Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.

--- Hoàn tất! Đã nạp 1000 dòng dữ liệu ---


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# **2.3.2.a Tokenization**
1. **Preprocessing Rationale (Lý do tiền xử lý):** Tách từ (word tokenization) và tách câu (sentence tokenization) là bước nền tảng trong NLP. Nó giúp chia nhỏ văn bản nguyên thủy thành các đơn vị ngữ nghĩa cơ bản (tokens), từ đó thuật toán mới có thể đếm tần suất và phân tích ngữ cảnh của từng từ.
2. **Implementation Details (Chi tiết triển khai):** Nhóm sử dụng thư viện NLTK với các hàm word_tokenize và sent_tokenize. Các hàm này xử lý dấu câu rất tốt, tách biệt dấu phẩy, dấu chấm ra khỏi từ một cách chính xác.
3. **Results and Analysis (Kết quả & Phân tích):** Phép tách từ cho phép ta thấy độ dài thực tế của câu (số lượng token), là cơ sở để thực hiện các bước làm sạch tiếp theo.

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

# Lấy thử 1 dòng đánh giá phim đầu tiên để demo
sample_text = df_sample['review'].iloc[0]

# 1. Tách câu (Sentence Tokenization)
sentences = sent_tokenize(sample_text)

# 2. Tách từ (Word Tokenization)
words = word_tokenize(sentences[0])

print("--- VĂN BẢN GỐC ---")
print(sample_text[:150], "...\n")

print("--- KẾT QUẢ TÁCH CÂU ---")
print(f"Tổng số câu: {len(sentences)}")
print("Câu 1:", sentences[0], "\n")

print("--- KẾT QUẢ TÁCH TỪ (CỦA CÂU 1) ---")
print(words)

--- VĂN BẢN GỐC ---
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with ...

--- KẾT QUẢ TÁCH CÂU ---
Tổng số câu: 10
Câu 1: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. 

--- KẾT QUẢ TÁCH TỪ (CỦA CÂU 1) ---
['One', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'Oz', 'episode', 'you', "'ll", 'be', 'hooked', '.']


# **2.3.2.b Removing Stop Words**
1. **Preprocessing Rationale:** Từ dừng (Stop words) là những từ phổ biến như 'the', 'a', 'is', 'in'... Chúng xuất hiện dày đặc nhưng mang rất ít giá trị phân loại sắc thái văn bản (tích cực/tiêu cực). Loại bỏ chúng giúp giảm nhiễu và thu gọn đáng kể kích thước tập từ vựng.
2. **Implementation Details:** Nhóm sử dụng bộ từ điển tiếng Anh stopwords có sẵn của NLTK. Mã nguồn duyệt qua từng token đã được tách ở bước trước và chỉ giữ lại những từ không nằm trong danh sách stop words. Các token cũng được đưa về chữ thường (lowercase) để so sánh chính xác.
3. **Results and Analysis:** Số lượng từ trong câu giảm đi rõ rệt, giúp bộ nhớ máy tính được giải phóng mà không làm mất đi những "từ khóa" (keywords) quan trọng chứa nội dung đánh giá bộ phim.

In [ ]:
from nltk.corpus import stopwords
import string

# Lấy danh sách stop words tiếng Anh và các dấu câu cơ bản
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

# Chuyển về chữ thường và lọc từ
filtered_words = [
    word.lower() for word in words
    if word.lower() not in stop_words and word not in punctuation
]

print(f"Số từ gốc (bao gồm dấu câu): {len(words)}")
print(f"Số từ sau khi lọc stop words: {len(filtered_words)}\n")

print("--- DANH SÁCH TỪ SAU KHI LỌC ---")
print(filtered_words)

Số từ gốc (bao gồm dấu câu): 19
Số từ sau khi lọc stop words: 9

--- DANH SÁCH TỪ SAU KHI LỌC ---
['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', "'ll", 'hooked']


# **2.3.2.c Stemming and Lemmatization**
1. **Preprocessing Rationale:** Trong tiếng Anh, một từ có nhiều biến thể (vd: run, running, ran). Bước này đưa chúng về chung một gốc từ, giúp mô hình hiểu chúng mang cùng một nghĩa cơ bản.
2. **Implementation Details:** > - Stemming: Sử dụng PorterStemmer. Thuật toán này đơn giản là cắt bỏ phần đuôi từ (hậu tố). Nhanh nhưng đôi khi tạo ra những từ vô nghĩa.

**Lemmatization:** Sử dụng WordNetLemmatizer. Thuật toán này tra cứu từ điển ngữ pháp để đưa từ về dạng nguyên thể. Chậm hơn nhưng chính xác về mặt ngôn ngữ học.

3. **Results and Analysis:** Lemmatization cho kết quả tự nhiên hơn Stemming. Ví dụ: 'running' qua Stemming thành 'run', nhưng nếu từ có quy tắc phức tạp hơn, Stemming có thể cắt sai, trong khi Lemmatizer giữ được gốc từ chuẩn xác. Nhóm quyết định chọn Lemmatization cho bộ dữ liệu IMDB vì review phim cần hiểu đúng ngữ cảnh.

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Tạo một câu mẫu chứa nhiều biến thể ngữ pháp để test
test_words = ["running", "flies", "happily", "better", "cats", "studying"]

stemmed = [stemmer.stem(w) for w in test_words]
# Chú ý: lemmatizer cần biết loại từ (POS tag), mặc định nó hiểu là Danh từ (Noun)
# Ở đây ta demo đơn giản với động từ (verb = 'v')
lemmatized = [lemmatizer.lemmatize(w, pos='v') for w in test_words]

df_compare = pd.DataFrame({
    'Từ gốc': test_words,
    'Stemming (Cắt đuôi)': stemmed,
    'Lemmatization (Nguyên thể)': lemmatized
})

print(df_compare)

     Từ gốc Stemming (Cắt đuôi) Lemmatization (Nguyên thể)
0   running                 run                        run
1     flies                 fli                        fly
2   happily             happili                    happily
3    better              better                     better
4      cats                 cat                        cat
5  studying               studi                      study


# **2.3.2.d Text Vectorization (TF-IDF)**
1. **Preprocessing Rationale:** Các mô hình Machine Learning chỉ thao tác được với ma trận số, không đọc được chữ. Vector hóa là bước "dịch" văn bản thành ma trận số toán học. Nhóm chọn TF-IDF thay vì Bag-of-Words (BoW) vì TF-IDF không chỉ đếm số lần xuất hiện mà còn đánh giá được độ hiếm/độ quan trọng của từ đó trong toàn bộ tập dữ liệu.
2. **Implementation Details:** Sử dụng hàm TfidfVectorizer của thư viện scikit-learn. Giới hạn ma trận ở 1000 từ vựng xuất hiện nhiều nhất (max_features=1000) để tránh hiện tượng tràn RAM (out of memory) do tính chất ma trận thưa (sparsity).
3. **Results and Analysis:** Kết quả trả về là một "Ma trận thưa" (Sparse Matrix). Số chiều (Dimensionality) là (Số lượng bài review) x (Số lượng từ vựng). Hầu hết các ô trong ma trận chứa giá trị 0 vì một bài review không thể chứa toàn bộ từ điển.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Khởi tạo TF-IDF Vectorizer
# max_features: giới hạn lấy 1000 từ quan trọng nhất
# stop_words='english': TfidfVectorizer cũng có sẵn tính năng lọc từ dừng!
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')

# Fit và Transform trên toàn bộ cột 'review' của 1000 mẫu dữ liệu
# Lưu ý: Trong thực tế ta phải chạy bước c (lemmatize) cho toàn bộ dataset trước khi đưa vào TF-IDF
X_tfidf = tfidf.fit_transform(df_sample['review'])

print("--- KẾT QUẢ VECTOR HÓA ---")
print(f"Kích thước (Shape) của ma trận TF-IDF: {X_tfidf.shape}")
print("(Nghĩa là: 1000 bài review, mỗi bài được biểu diễn bằng 1000 con số)")
print(f"Độ thưa thớt (Kiểu dữ liệu): {type(X_tfidf)}\n")

# In thử tên của 10 cột (10 từ vựng) ngẫu nhiên mà thuật toán đã học được
print("10 từ khóa ngẫu nhiên trong danh sách features:")
print(tfidf.get_feature_names_out()[200:210])

--- KẾT QUẢ VECTOR HÓA ---
Kích thước (Shape) của ma trận TF-IDF: (1000, 1000)
(Nghĩa là: 1000 bài review, mỗi bài được biểu diễn bằng 1000 con số)
Độ thưa thớt (Kiểu dữ liệu): <class 'scipy.sparse._csr.csr_matrix'>

10 từ khóa ngẫu nhiên trong danh sách features:
['decides' 'deep' 'definitely' 'delivers' 'depth' 'deserves' 'despite'
 'details' 'detective' 'developed']
